# Tema 2 — Integrare VLM (BiomedCLIP) pe Kaggle

**Setari (panoul din dreapta):** Accelerator = **GPU**, Internet = **On**,
Add Input = datasetul de mamografie + datasetul cu cod (`breastcancer-vlm-code`).

Ruleaza celulele in ordine, de sus in jos.


In [ ]:
# 0. GPU + ce e atasat
import torch, os
print('CUDA:', torch.cuda.is_available(), '-', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
print('Input-uri in /kaggle/input:')
for d in sorted(os.listdir('/kaggle/input')): print('  ', d)

In [ ]:
# 1. Dependinte (~2-3 min). transformers e OBLIGATORIU pentru textul BiomedCLIP.
!pip -q install open_clip_torch transformers timm efficientnet_pytorch torchmetrics albumentations
!pip -q install 'git+https://github.com/facebookresearch/segment-anything.git'
print('gata dependinte')

In [ ]:
# 2. Gaseste codul (pachetul vlm/) si copiaza-l in /kaggle/working/code
import os, shutil, sys
code_src = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'vlm' in dirs and 'PatchFilterClassificationTrainingAndTesting.py' in files:
        code_src = root; break
assert code_src, 'Nu am gasit codul. Ai atasat datasetul breastcancer-vlm-code?'
CODE_DIR = '/kaggle/working/code'
if os.path.exists(CODE_DIR): shutil.rmtree(CODE_DIR)
shutil.copytree(code_src, CODE_DIR)
os.chdir(CODE_DIR); sys.path.insert(0, CODE_DIR)
print('cod:', code_src, '->', CODE_DIR)

In [ ]:
# 3. Descopera structura datasetului
import os
def cpng(d):
    try: return sum(1 for f in os.listdir(d) if f.lower().endswith('.png'))
    except Exception: return 0
dirs = []
for root, _, files in os.walk('/kaggle/input'):
    n = sum(1 for f in files if f.lower().endswith('.png'))
    if n >= 5: dirs.append((n, root))
dirs.sort(reverse=True)
print('Foldere cu .png (nr, cale):')
for n, d in dirs[:12]: print(f'  {n:6d}  {d}')

## CONFIG — caile sunt pre-completate cu ce ai acum.
Daca celula 3 arata alte cai, ajusteaza-le aici.

In [ ]:
# cai (pre-completate; verifica cu celula 3)
IMG_DIR   = '/kaggle/input/datasets/rbogdan334cb/mammo-dataset/InBreast/Normal'
MASK_DIR  = '/kaggle/input/datasets/rbogdan334cb/mammo-dataset/InBreast/Masks'
CALC_MASK = '/kaggle/input/datasets/rbogdan334cb/mammo-dataset/InBreast/Masks/Calcification_Only'
FOLDS_CSV = f'{CODE_DIR}/folds.csv'

RETINA_CKPT = ''    # pt. detectie (ex: /kaggle/input/<ckpts>/retina_detection_best_model.pth)
MEDSAM_CKPT = ''    # pt. segmentare (ex: /kaggle/input/<ckpts>/medsam_best.pth)

PATCH_SIZE = 128
FOLD       = 1

for name, p in [('IMG_DIR', IMG_DIR), ('CALC_MASK', CALC_MASK), ('FOLDS_CSV', FOLDS_CSV)]:
    print(('OK   ' if os.path.exists(p) else 'LIPSA'), name, '->', p)

In [ ]:
# 4. Sanity: testele de logica (rapide, fara model)
!python vlm/verify_logic.py | tail -n 3

## Etapa 1 — Clasificare (scrisa inline; ~15 min)
Zero-shot VLM + fuziune + comparatie corecta (acelasi cap antrenat pe CNN / VLM / fuziune).

In [ ]:
# ===== CLASIFICARE (self-contained) =====
import sys, os, json, numpy as np, cv2, torch, torch.nn as nn, torch.nn.functional as F
sys.path.insert(0, CODE_DIR)
from vlm.vlm_core import VLM
from vlm.clinical_prompts import CLASS_PROMPTS
from vlm.fusion_ops import late_fusion, sweep_alpha, agreement_analysis
from PatchFilterClassificationTrainingAndTesting import extract_patches_from_dataset, seed_everything
from sklearn.model_selection import train_test_split
import timm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
seed_everything(42)

data, num_cls = extract_patches_from_dataset(IMG_DIR, MASK_DIR, PATCH_SIZE, overlap=True, balance_mode='balanced')
patches = [d['patch'] for d in data]; labels = [d['label'] for d in data]
tr_i, va_i = train_test_split(np.arange(len(patches)), test_size=0.2, random_state=42, stratify=labels)

class CNNFeat(nn.Module):
    def __init__(self, ncls):
        super().__init__(); self.net = timm.create_model('resnet50', pretrained=True, num_classes=ncls)
    def forward(self, x):
        f = self.net.forward_head(self.net.forward_features(x), pre_logits=True)
        return self.net.get_classifier()(f), f

def prep(patch, size=224):
    p = cv2.cvtColor(patch, cv2.COLOR_GRAY2RGB).astype(np.float32) / 255.0
    p = cv2.resize(p, (size, size))
    p = (p - np.array([0.485, 0.456, 0.406], np.float32)) / np.array([0.229, 0.224, 0.225], np.float32)
    return torch.from_numpy(p.transpose(2, 0, 1))

@torch.no_grad()
def featurize(cnn, vlm, pts, lbls, bs=64):
    keys, bank = vlm.build_text_bank(CLASS_PROMPTS)
    cp, cf, ve, vp = [], [], [], []
    for i in range(0, len(pts), bs):
        ch = pts[i:i+bs]
        x = torch.stack([prep(p) for p in ch]).to(DEVICE)
        lo, fe = cnn(x)
        cp.append(F.softmax(lo, 1).cpu().numpy()); cf.append(fe.cpu().numpy())
        e = vlm.encode_image(ch); ve.append(e.cpu().numpy())
        vp.append(vlm.similarity(e, bank, softmax=True).cpu().numpy())
    return {'cnn_probs': np.concatenate(cp), 'cnn_feat': np.concatenate(cf),
            'vlm_emb': np.concatenate(ve), 'vlm_probs': np.concatenate(vp), 'labels': np.asarray(lbls)}

print('loading VLM (BiomedCLIP) + CNN ...')
vlm = VLM(device=DEVICE); cnn = CNNFeat(num_cls).to(DEVICE).eval()
print('featurizing (~15 min)...')
tr = featurize(cnn, vlm, [patches[i] for i in tr_i], [labels[i] for i in tr_i])
va = featurize(cnn, vlm, [patches[i] for i in va_i], [labels[i] for i in va_i])
yva = va['labels']

def train_head(Xtr, ytr, Xva, yv, ncls, epochs=40, bs=256):
    h = nn.Sequential(nn.Linear(Xtr.shape[1], 512), nn.BatchNorm1d(512), nn.ReLU(True),
                      nn.Dropout(0.3), nn.Linear(512, ncls)).to(DEVICE)
    opt = torch.optim.AdamW(h.parameters(), lr=1e-3, weight_decay=1e-4)
    Xt = torch.tensor(Xtr, dtype=torch.float32); yt = torch.tensor(ytr, dtype=torch.long)
    Xv = torch.tensor(Xva, dtype=torch.float32).to(DEVICE)
    best, bp = -1, None
    for _ in range(epochs):
        h.train(); order = torch.randperm(len(Xt))
        for i in range(0, len(order), bs):
            j = order[i:i+bs]; opt.zero_grad()
            loss = F.cross_entropy(h(Xt[j].to(DEVICE)), yt[j].to(DEVICE)); loss.backward(); opt.step()
        h.eval()
        with torch.no_grad(): p = F.softmax(h(Xv), 1).cpu().numpy()
        a = (p.argmax(1) == yv).mean()
        if a > best: best, bp = a, p
    return bp

acc = lambda p, y: float((p.argmax(1) == y).mean())
res = {}
res['cnn_only_raw'] = acc(va['cnn_probs'], yva)
res['vlm_zeroshot'] = acc(va['vlm_probs'], yva)
(ba, _), _ = sweep_alpha(tr['cnn_probs'], tr['vlm_probs'], tr['labels'])
res['late_fusion'] = acc(late_fusion(va['cnn_probs'], va['vlm_probs'], ba), yva)
ct = np.concatenate([tr['cnn_feat'], tr['vlm_emb']], 1); cv = np.concatenate([va['cnn_feat'], va['vlm_emb']], 1)
res['cnn_trained'] = acc(train_head(tr['cnn_feat'], tr['labels'], va['cnn_feat'], yva, num_cls), yva)
res['vlm_trained'] = acc(train_head(tr['vlm_emb'], tr['labels'], va['vlm_emb'], yva, num_cls), yva)
res['feature_fusion'] = acc(train_head(ct, tr['labels'], cv, yva, num_cls), yva)

print('\n===== REZULTATE CLASIFICARE =====')
for k in ['cnn_only_raw', 'vlm_zeroshot', 'late_fusion', 'cnn_trained', 'vlm_trained', 'feature_fusion']:
    print(f'  {k:16s} acc = {res[k]:.4f}')
print(f"\n>> Contributia marginala VLM (feature_fusion - cnn_trained) = {res['feature_fusion'] - res['cnn_trained']:+.4f}")
ag = agreement_analysis(va['cnn_probs'], va['vlm_probs'], yva); ag.pop('difficulty')
print('\nAcord CNN<->VLM (dificultate):', json.dumps(ag, indent=2))
os.makedirs('/kaggle/working/VLM_Classification_Results', exist_ok=True)
json.dump({**res, 'agreement': ag}, open('/kaggle/working/VLM_Classification_Results/summary.json', 'w'), indent=2)
print('\nsalvat -> /kaggle/working/VLM_Classification_Results/summary.json')

## Etapa 2 — Detectie  (necesita RETINA_CKPT + folds.csv)
Ruleaza doar dupa ce ai pus calea catre retina_detection_best_model.pth in CONFIG.

In [ ]:
os.environ.update({
    'SOURCE_IMG_DIR': IMG_DIR, 'SOURCE_MASK_DIR': CALC_MASK, 'FOLDS_CSV': FOLDS_CSV,
    'ENCODER_INIT': 'imagenet', 'CKPT': RETINA_CKPT, 'FOLD': str(FOLD),
    'BETA': '0.5', 'MIN_VLM': '0.35', 'RECOVER': '1',
    'OUT_DIR': '/kaggle/working/VLM_Detection_Results'})
if RETINA_CKPT:
    !python vlm/detection_vlm.py
    import json; print(json.dumps(json.load(open('/kaggle/working/VLM_Detection_Results/detection_vlm_summary.json')), indent=2))
else:
    print('Sari peste: nu ai setat RETINA_CKPT in CONFIG.')

## Etapa 3 — Segmentare  (necesita MEDSAM_CKPT + folds.csv)

In [ ]:
os.environ.update({
    'SOURCE_IMG_DIR': IMG_DIR, 'SOURCE_MASK_DIR': CALC_MASK, 'FOLDS_CSV': FOLDS_CSV,
    'CKPT': MEDSAM_CKPT, 'SAM_BASE': '', 'FOLD': str(FOLD), 'REL_THR': '0.5',
    'OUT_DIR': '/kaggle/working/VLM_Segmentation_Results'})
if MEDSAM_CKPT:
    !python vlm/segmentation_vlm.py
    import json; print(json.dumps(json.load(open('/kaggle/working/VLM_Segmentation_Results/segmentation_vlm_summary.json')), indent=2))
else:
    print('Sari peste: nu ai setat MEDSAM_CKPT in CONFIG.')